In [1]:
import json, random
import pandas as pd

d = json.load(open("../data/CUADv1.json"))["data"]
print(len(d), "contracts")

rows = []
for doc in d:
    p = doc["paragraphs"][0]
    for qa in p["qas"]:
        rows.append({
            "contract": doc["title"],
            "clause": qa["id"].split("__")[-1],
            "present": not qa["is_impossible"],
            "answers": [a["text"] for a in qa["answers"]],
            "starts": [a["answer_start"] for a in qa["answers"]],
        })
df = pd.DataFrame(rows)
texts = {doc["title"]: doc["paragraphs"][0]["context"] for doc in d}
print(df["clause"].nunique(), "clause types")

510 contracts
41 clause types


In [2]:
KEEP = [
    "Cap On Liability", "Uncapped Liability", "Renewal Term",
    "Notice Period To Terminate Renewal", "Termination For Convenience",
    "Non-Compete", "Exclusivity", "Ip Ownership Assignment",
    "Governing Law", "Anti-Assignment",
]
all_clauses = {c.lower(): c for c in df["clause"].unique()}
missing = [k for k in KEEP if k.lower() not in all_clauses]
print("Not found:", missing)

KEEP = [all_clauses[k.lower()] for k in KEEP if k.lower() in all_clauses]
df10 = df[df["clause"].isin(KEEP)].copy()

Not found: []


In [3]:
stats = df10.groupby("clause")["present"].agg(present="sum", total="count")
stats["pct"] = (stats["present"] / stats["total"] * 100).round(1)
stats.sort_values("present")

,present,total,pct
clause,,,
Notice Period To Terminate Renewal,111,510,21.8
Uncapped Liability,111,510,21.8
Non-Compete,119,510,23.3
Ip Ownership Assignment,124,510,24.3
Renewal Term,176,510,34.5
Exclusivity,180,510,35.3
Termination For Convenience,183,510,35.9
Cap On Liability,275,510,53.9
Anti-Assignment,374,510,73.3


In [4]:
lengths = pd.Series({t: len(x.split()) for t, x in texts.items()})
lengths.describe()

count      510.000000
mean      7861.188235
std       8367.292546
min        109.000000
25%       2457.000000
50%       5006.000000
75%      10196.250000
max      47733.000000
dtype: float64

In [5]:
titles = sorted(texts)
random.seed(42)
test_ids = random.sample(titles, 100)
dev_ids = [t for t in titles if t not in set(test_ids)]

json.dump({"test": test_ids, "dev": dev_ids}, open("../data/split.json", "w"), indent=1)
print(len(dev_ids), "dev,", len(test_ids), "test")

410 dev, 100 test


In [6]:
t = dev_ids[0]
print(texts[t][:3000])
df10[df10["contract"] == t][["clause", "present", "answers"]]

CO-BRANDING AND ADVERTISING AGREEMENT

THIS CO-BRANDING AND ADVERTISING AGREEMENT (the "Agreement") is made as of June 21, 1999 (the "Effective Date") by and between I-ESCROW, INC., with its principal place of business at 1730 S. Amphlett Blvd., Suite 233, San Mateo, California 94402 ("i-Escrow"), and 2THEMART.COM, INC. having its principal place of business at 18301 Von Karman Avenue, 7th Floor, Irvine, California 92612 ("2TheMart").

1. DEFINITIONS.

(a) "CONTENT" means all content or information, in any medium, provided by a party to the other party for use in conjunction with the performance of its obligations hereunder, including without limitation any text, music, sound, photographs, video, graphics, data or software. Content provided by 2TheMart is referred to herein as "2TheMart Content" and Content provided by i-Escrow is referred to herein as "i-Escrow Content."

(b) "CO-BRANDED SITE" means the web-site accessible through Domain Name, for the Services implemented by i-Escrow.

,clause,present,answers
18414,Renewal Term,True,[A party wishing to renew this Agreement shall...
18415,Notice Period To Terminate Renewal,True,[A party wishing to renew this Agreement shall...
18416,Governing Law,True,[This Agreement will be governed and construed...
18418,Non-Compete,True,[i-Escrow shall not run banner advertisements ...
18419,Exclusivity,False,[]
18424,Termination For Convenience,False,[]
18427,Anti-Assignment,True,[All rights (under any applicable intellectual...
18432,Ip Ownership Assignment,False,[]
18443,Uncapped Liability,True,[EXCEPT IN THE EVENT OF A BREACH OF SECTION 11...
18444,Cap On Liability,True,[EXCEPT IN THE EVENT OF A BREACH OF SECTION 11...
